# RadonPy Interactive Tutorial 🧪

## 聚合物材料自动化MD模拟与可视化教程

[![GitHub](https://img.shields.io/badge/GitHub-RadonPy-blue)](https://github.com/RadonPy/RadonPy)
[![Version](https://img.shields.io/badge/Version-0.2.10-green)]()
[![Python](https://img.shields.io/badge/Python-3.7--3.12-yellow)]()

这个交互式教程将带你一步步学习如何使用 RadonPy 进行聚合物材料的分子动力学模拟。

### 📚 教程内容
1. 环境设置和导入
2. 分子结构可视化
3. 聚合物链生成与可视化
4. 力场参数分析
5. MD模拟流程
6. 结果分析与可视化
7. 机器学习预测

## 1. 环境设置和导入 🚀

In [ ]:
# 安装必要的包（如果还没有安装）
# !pip install radonpy-pypi rdkit matplotlib pandas numpy scipy seaborn plotly

# 导入基础库
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# 数据处理和可视化
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML, Image

# 设置绘图风格
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# RadonPy 核心模块
from radonpy.core import utils, poly
from radonpy.ff.gaff2_mod import GAFF2_mod
from radonpy.sim import qm
from radonpy.sim.preset import eq, tc

# RDKit 用于分子可视化
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem
from rdkit.Chem.Draw import IPythonConsole

print("✅ 所有库导入成功！")
print(f"Python版本: {sys.version}")
print(f"工作目录: {os.getcwd()}")

## 2. 分子结构可视化 🔬

让我们从几个常见聚合物的重复单元开始：

In [ ]:
# 定义聚合物数据库
polymers_db = {
    'PE': {'smiles': '*CC(*)', 'name': '聚乙烯', 'color': '#FF6B6B'},
    'PP': {'smiles': '*CC(*)(C)', 'name': '聚丙烯', 'color': '#4ECDC4'},
    'PS': {'smiles': '*CC(*)c1ccccc1', 'name': '聚苯乙烯', 'color': '#45B7D1'},
    'PVC': {'smiles': '*CC(*)Cl', 'name': '聚氯乙烯', 'color': '#96CEB4'},
    'PMMA': {'smiles': '*CC(*)(C)C(=O)OC', 'name': '聚甲基丙烯酸甲酯', 'color': '#FFEAA7'},
    'PTFE': {'smiles': '*C(F)C(*)F', 'name': '聚四氟乙烯', 'color': '#DDA0DD'},
}

# 创建分子对象并可视化
def visualize_monomers(polymers_db):
    """可视化聚合物重复单元"""
    mols = []
    legends = []
    
    for code, info in polymers_db.items():
        mol = utils.mol_from_smiles(info['smiles'])
        mols.append(mol)
        legends.append(f"{code}: {info['name']}")
    
    # 绘制分子网格
    img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 300), 
                               legends=legends, returnPNG=False)
    return img

# 显示分子结构
print("📊 常见聚合物重复单元结构：")
visualize_monomers(polymers_db)

### 2.1 分子性质计算与可视化

In [ ]:
# 计算分子描述符
def calculate_molecular_properties(polymers_db):
    """计算并可视化分子性质"""
    properties = []
    
    for code, info in polymers_db.items():
        mol = utils.mol_from_smiles(info['smiles'])
        
        props = {
            'Polymer': code,
            'Name': info['name'],
            'MW': Descriptors.ExactMolWt(mol),
            'HeavyAtoms': mol.GetNumHeavyAtoms(),
            'Bonds': mol.GetNumBonds(),
            'RotatableBonds': Descriptors.NumRotatableBonds(mol),
            'HBD': Descriptors.NumHDonors(mol),
            'HBA': Descriptors.NumHAcceptors(mol),
            'TPSA': Descriptors.TPSA(mol),
            'LogP': Descriptors.MolLogP(mol)
        }
        properties.append(props)
    
    df = pd.DataFrame(properties)
    return df

# 计算性质
props_df = calculate_molecular_properties(polymers_db)

# 创建交互式表格
from IPython.display import display
styled_df = props_df.style.background_gradient(subset=['MW', 'LogP', 'TPSA'])
display(HTML("<h4>分子性质对比表</h4>"))
display(styled_df)

In [ ]:
# 创建雷达图比较聚合物性质
import plotly.graph_objects as go

def create_polymer_radar_chart(props_df):
    """创建聚合物性质雷达图"""
    
    # 标准化数据
    properties = ['HeavyAtoms', 'Bonds', 'RotatableBonds', 'HBA', 'LogP']
    
    fig = go.Figure()
    
    for _, row in props_df.iterrows():
        values = []
        for prop in properties:
            if prop in row:
                # 标准化到0-1范围
                max_val = props_df[prop].max()
                min_val = props_df[prop].min()
                if max_val != min_val:
                    normalized = (row[prop] - min_val) / (max_val - min_val)
                else:
                    normalized = 0.5
                values.append(normalized)
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=properties,
            fill='toself',
            name=row['Polymer']
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]
            )),
        showlegend=True,
        title="聚合物性质雷达图对比"
    )
    
    return fig

# 显示雷达图
radar_fig = create_polymer_radar_chart(props_df)
radar_fig.show()

## 3. 聚合物链生成与可视化 🔗

现在让我们生成实际的聚合物链并可视化其3D结构：

In [ ]:
# 选择聚苯乙烯作为示例
polymer_type = 'PS'
smiles = polymers_db[polymer_type]['smiles']
print(f"🔬 生成{polymers_db[polymer_type]['name']}链...")

# 创建重复单元
monomer = utils.mol_from_smiles(smiles)
terminal = utils.mol_from_smiles('*C')

# 生成不同聚合度的链
degrees_of_polymerization = [5, 10, 20]
polymer_chains = {}

for dp in degrees_of_polymerization:
    print(f"  生成DP={dp}的聚合物链...")
    chain = poly.polymerize_rw(monomer, dp, tacticity='atactic')
    chain = poly.terminate_rw(chain, terminal)
    polymer_chains[dp] = chain
    
    # 计算链的性质
    n_atoms = chain.GetNumAtoms()
    n_bonds = chain.GetNumBonds()
    mw = Descriptors.ExactMolWt(chain)
    
    print(f"    ✓ 原子数: {n_atoms}, 键数: {n_bonds}, 分子量: {mw:.2f}")

print("\n✅ 聚合物链生成完成！")

In [ ]:
# 可视化聚合物链的回转半径随聚合度的变化
def calculate_radius_of_gyration(mol):
    """计算回转半径（简化版）"""
    conf = mol.GetConformer()
    positions = conf.GetPositions()
    center = positions.mean(axis=0)
    distances = np.linalg.norm(positions - center, axis=1)
    rg = np.sqrt(np.mean(distances**2))
    return rg

# 计算不同链长的回转半径
rg_data = []
for dp, chain in polymer_chains.items():
    AllChem.EmbedMolecule(chain, randomSeed=42)
    AllChem.UFFOptimizeMolecule(chain)
    rg = calculate_radius_of_gyration(chain)
    rg_data.append({'DP': dp, 'Rg': rg})

rg_df = pd.DataFrame(rg_data)

# 绘制回转半径vs聚合度
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 线性图
ax1.plot(rg_df['DP'], rg_df['Rg'], 'o-', markersize=10, linewidth=2)
ax1.set_xlabel('聚合度 (DP)', fontsize=12)
ax1.set_ylabel('回转半径 (Å)', fontsize=12)
ax1.set_title('回转半径 vs 聚合度', fontsize=14)
ax1.grid(True, alpha=0.3)

# 对数图（检查标度关系）
ax2.loglog(rg_df['DP'], rg_df['Rg'], 'o-', markersize=10, linewidth=2)
ax2.set_xlabel('log(DP)', fontsize=12)
ax2.set_ylabel('log(Rg)', fontsize=12)
ax2.set_title('标度关系: Rg ~ DP^ν', fontsize=14)
ax2.grid(True, alpha=0.3)

# 拟合幂律
from scipy.optimize import curve_fit
def power_law(x, a, b):
    return a * x**b

popt, _ = curve_fit(power_law, rg_df['DP'], rg_df['Rg'])
x_fit = np.linspace(5, 20, 100)
y_fit = power_law(x_fit, *popt)
ax2.plot(x_fit, y_fit, 'r--', alpha=0.7, label=f'Rg ~ DP^{popt[1]:.2f}')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n📊 标度指数 ν = {popt[1]:.3f}")
print(f"   (理想链: ν=0.5, 良溶剂: ν=0.6, 塌缩: ν=0.33)")

## 4. 力场参数分析 ⚙️

让我们分析力场参数并可视化其分布：

In [ ]:
# 为聚合物分配力场参数
print("🔧 分配GAFF2_mod力场参数...")

ff = GAFF2_mod()
test_polymer = polymer_chains[10]  # 使用DP=10的链

# 分配力场
success = ff.ff_assign(test_polymer)
if success:
    print("✅ 力场分配成功！")
else:
    print("❌ 力场分配失败！")

# 提取力场参数
def extract_ff_parameters(mol):
    """提取力场参数用于可视化"""
    params = {
        'atom_types': [],
        'charges': [],
        'bond_params': [],
        'angle_params': [],
    }
    
    # 原子参数
    for atom in mol.GetAtoms():
        if atom.HasProp('ff_type'):
            params['atom_types'].append(atom.GetProp('ff_type'))
        if atom.HasProp('ff_charge'):
            params['charges'].append(float(atom.GetProp('ff_charge')))
    
    # 键参数
    for bond in mol.GetBonds():
        if bond.HasProp('ff_k'):
            params['bond_params'].append({
                'k': float(bond.GetProp('ff_k')),
                'r0': float(bond.GetProp('ff_r0'))
            })
    
    return params

ff_params = extract_ff_parameters(test_polymer)

# 统计原子类型
from collections import Counter
atom_type_counts = Counter(ff_params['atom_types'])

print(f"\n📊 原子类型分布:")
for atype, count in atom_type_counts.most_common():
    print(f"   {atype}: {count}个")

In [ ]:
# 可视化力场参数分布
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 原子类型饼图
ax1 = axes[0, 0]
colors = plt.cm.Set3(np.linspace(0, 1, len(atom_type_counts)))
wedges, texts, autotexts = ax1.pie(atom_type_counts.values(), 
                                     labels=atom_type_counts.keys(),
                                     colors=colors,
                                     autopct='%1.1f%%',
                                     startangle=90)
ax1.set_title('原子类型分布', fontsize=14, fontweight='bold')

# 2. 电荷分布直方图
ax2 = axes[0, 1]
if ff_params['charges']:
    ax2.hist(ff_params['charges'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax2.set_xlabel('原子电荷 (e)', fontsize=12)
    ax2.set_ylabel('频数', fontsize=12)
    ax2.set_title('原子电荷分布', fontsize=14, fontweight='bold')
    ax2.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='中性')
    ax2.legend()
    
    # 添加统计信息
    mean_charge = np.mean(ff_params['charges'])
    std_charge = np.std(ff_params['charges'])
    ax2.text(0.05, 0.95, f'平均值: {mean_charge:.3f}\n标准差: {std_charge:.3f}',
             transform=ax2.transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 3. 键长参数分布
ax3 = axes[1, 0]
if ff_params['bond_params']:
    bond_r0 = [p['r0'] for p in ff_params['bond_params']]
    ax3.hist(bond_r0, bins=15, alpha=0.7, color='lightgreen', edgecolor='black')
    ax3.set_xlabel('平衡键长 (Å)', fontsize=12)
    ax3.set_ylabel('频数', fontsize=12)
    ax3.set_title('键长参数分布', fontsize=14, fontweight='bold')

# 4. 键力常数分布
ax4 = axes[1, 1]
if ff_params['bond_params']:
    bond_k = [p['k'] for p in ff_params['bond_params']]
    ax4.hist(bond_k, bins=15, alpha=0.7, color='salmon', edgecolor='black')
    ax4.set_xlabel('力常数 (kcal/mol/Å²)', fontsize=12)
    ax4.set_ylabel('频数', fontsize=12)
    ax4.set_title('键力常数分布', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. MD模拟流程监控 📈

模拟平衡过程并实时监控关键参数：

In [ ]:
# 模拟MD数据（实际运行时会产生真实数据）
def simulate_md_trajectory(n_steps=5000, dt=100):
    """模拟MD轨迹数据用于演示"""
    time = np.arange(0, n_steps, dt)
    
    # 模拟温度（逐渐收敛到目标温度）
    target_temp = 300
    temp = target_temp + 50 * np.exp(-time/1000) * np.sin(time/100) + np.random.normal(0, 5, len(time))
    
    # 模拟压力
    target_press = 1.0
    press = target_press + 0.5 * np.exp(-time/1500) * np.cos(time/150) + np.random.normal(0, 0.1, len(time))
    
    # 模拟密度（逐渐收敛）
    target_density = 1.05
    density = target_density - 0.2 * np.exp(-time/2000) + np.random.normal(0, 0.01, len(time))
    
    # 模拟总能量
    energy = -10000 + 500 * np.exp(-time/1000) + np.random.normal(0, 50, len(time))
    
    # 模拟回转半径
    rg = 15 + 2 * np.sin(time/500) * np.exp(-time/3000) + np.random.normal(0, 0.2, len(time))
    
    df = pd.DataFrame({
        'Time': time,
        'Temperature': temp,
        'Pressure': press,
        'Density': density,
        'Energy': energy,
        'Rg': rg
    })
    
    return df

# 生成模拟数据
md_data = simulate_md_trajectory()
print("📊 生成了模拟MD轨迹数据")
print(md_data.head())

In [ ]:
# 创建交互式MD监控仪表板
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def create_md_dashboard(md_data):
    """创建MD模拟监控仪表板"""
    
    # 创建子图
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=('温度演化', '压力演化', 
                       '密度收敛', '能量变化',
                       '回转半径', '平衡判定'),
        vertical_spacing=0.12,
        horizontal_spacing=0.15
    )
    
    # 温度
    fig.add_trace(
        go.Scatter(x=md_data['Time'], y=md_data['Temperature'],
                  mode='lines', name='温度',
                  line=dict(color='red', width=1)),
        row=1, col=1
    )
    fig.add_hline(y=300, line_dash="dash", line_color="darkred", 
                  annotation_text="目标: 300K", row=1, col=1)
    
    # 压力
    fig.add_trace(
        go.Scatter(x=md_data['Time'], y=md_data['Pressure'],
                  mode='lines', name='压力',
                  line=dict(color='blue', width=1)),
        row=1, col=2
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="darkblue",
                  annotation_text="目标: 1 atm", row=1, col=2)
    
    # 密度
    fig.add_trace(
        go.Scatter(x=md_data['Time'], y=md_data['Density'],
                  mode='lines', name='密度',
                  line=dict(color='green', width=1)),
        row=2, col=1
    )
    
    # 添加移动平均线
    window = 10
    density_ma = md_data['Density'].rolling(window=window).mean()
    fig.add_trace(
        go.Scatter(x=md_data['Time'], y=density_ma,
                  mode='lines', name='移动平均',
                  line=dict(color='darkgreen', width=2)),
        row=2, col=1
    )
    
    # 能量
    fig.add_trace(
        go.Scatter(x=md_data['Time'], y=md_data['Energy'],
                  mode='lines', name='总能量',
                  line=dict(color='purple', width=1)),
        row=2, col=2
    )
    
    # 回转半径
    fig.add_trace(
        go.Scatter(x=md_data['Time'], y=md_data['Rg'],
                  mode='lines', name='Rg',
                  line=dict(color='orange', width=1)),
        row=3, col=1
    )
    
    # 平衡判定指标
    # 计算变异系数（CV）作为平衡指标
    window_size = 20
    cv_density = md_data['Density'].rolling(window=window_size).std() / md_data['Density'].rolling(window=window_size).mean()
    cv_energy = md_data['Energy'].rolling(window=window_size).std() / np.abs(md_data['Energy'].rolling(window=window_size).mean())
    
    fig.add_trace(
        go.Scatter(x=md_data['Time'], y=cv_density,
                  mode='lines', name='密度CV',
                  line=dict(color='teal', width=2)),
        row=3, col=2
    )
    
    fig.add_trace(
        go.Scatter(x=md_data['Time'], y=cv_energy,
                  mode='lines', name='能量CV',
                  line=dict(color='brown', width=2)),
        row=3, col=2
    )
    
    fig.add_hline(y=0.01, line_dash="dash", line_color="gray",
                  annotation_text="平衡阈值", row=3, col=2)
    
    # 更新布局
    fig.update_xaxes(title_text="时间 (ps)", row=3, col=1)
    fig.update_xaxes(title_text="时间 (ps)", row=3, col=2)
    
    fig.update_yaxes(title_text="温度 (K)", row=1, col=1)
    fig.update_yaxes(title_text="压力 (atm)", row=1, col=2)
    fig.update_yaxes(title_text="密度 (g/cm³)", row=2, col=1)
    fig.update_yaxes(title_text="能量 (kcal/mol)", row=2, col=2)
    fig.update_yaxes(title_text="Rg (Å)", row=3, col=1)
    fig.update_yaxes(title_text="变异系数", row=3, col=2)
    
    fig.update_layout(
        height=900,
        showlegend=True,
        title_text="MD模拟实时监控仪表板",
        title_font_size=20
    )
    
    return fig

# 创建并显示仪表板
dashboard = create_md_dashboard(md_data)
dashboard.show()

## 6. 物性计算结果分析 📊

分析和可视化计算得到的物理性质：

In [ ]:
# 模拟多个聚合物的物性数据
def generate_polymer_properties():
    """生成聚合物物性数据库"""
    
    properties_data = []
    
    # 基于文献的典型值范围
    polymer_properties = {
        'PE': {'density': 0.95, 'Tg': -120, 'Tm': 135, 'E': 1.0, 'tc': 0.5},
        'PP': {'density': 0.90, 'Tg': -10, 'Tm': 165, 'E': 1.5, 'tc': 0.2},
        'PS': {'density': 1.05, 'Tg': 100, 'Tm': 240, 'E': 3.0, 'tc': 0.15},
        'PVC': {'density': 1.40, 'Tg': 80, 'Tm': 212, 'E': 3.5, 'tc': 0.19},
        'PMMA': {'density': 1.18, 'Tg': 105, 'Tm': 160, 'E': 3.0, 'tc': 0.19},
        'PTFE': {'density': 2.20, 'Tg': -97, 'Tm': 327, 'E': 0.5, 'tc': 0.25},
    }
    
    for code, ref_props in polymer_properties.items():
        # 添加随机变化模拟实验误差
        props = {
            'Polymer': code,
            'Name': polymers_db[code]['name'],
            'Density': ref_props['density'] + np.random.normal(0, 0.02),
            'Tg': ref_props['Tg'] + np.random.normal(0, 5),
            'Tm': ref_props['Tm'] + np.random.normal(0, 5),
            'E_modulus': ref_props['E'] + np.random.normal(0, 0.1),
            'Thermal_Cond': ref_props['tc'] + np.random.normal(0, 0.02),
            'Cp': 1800 + np.random.normal(0, 100),  # J/kg·K
            'CTE': (65 + np.random.normal(0, 5)) * 1e-6,  # 1/K
            'Dielectric': 2.5 + np.random.normal(0, 0.1),
        }
        properties_data.append(props)
    
    return pd.DataFrame(properties_data)

# 生成数据
polymer_props_df = generate_polymer_properties()

# 显示数据表格
display(HTML("<h3>🧪 聚合物物性计算结果</h3>"))
display(polymer_props_df.style.format({
    'Density': '{:.3f}',
    'Tg': '{:.1f}',
    'Tm': '{:.1f}',
    'E_modulus': '{:.2f}',
    'Thermal_Cond': '{:.3f}',
    'Cp': '{:.0f}',
    'CTE': '{:.2e}',
    'Dielectric': '{:.2f}'
}).background_gradient(subset=['Density', 'Tg', 'E_modulus', 'Thermal_Cond']))

In [ ]:
# 创建物性相关性热图
import seaborn as sns

# 选择数值列
numeric_cols = ['Density', 'Tg', 'Tm', 'E_modulus', 'Thermal_Cond', 'Cp', 'Dielectric']
corr_matrix = polymer_props_df[numeric_cols].corr()

# 创建热图
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title('聚合物物性相关性矩阵', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# 找出强相关性
strong_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            strong_corr.append({
                'Property 1': corr_matrix.columns[i],
                'Property 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

if strong_corr:
    print("\n🔍 强相关性 (|r| > 0.7):")
    for item in strong_corr:
        print(f"   {item['Property 1']} ↔ {item['Property 2']}: {item['Correlation']:.3f}")

In [ ]:
# 创建3D散点图展示多维物性关系
import plotly.express as px

fig = px.scatter_3d(polymer_props_df, 
                    x='Density', 
                    y='Tg', 
                    z='E_modulus',
                    color='Thermal_Cond',
                    size='Cp',
                    hover_data=['Name'],
                    text='Polymer',
                    title='聚合物物性3D可视化',
                    labels={
                        'Density': '密度 (g/cm³)',
                        'Tg': '玻璃化温度 (°C)',
                        'E_modulus': '弹性模量 (GPa)',
                        'Thermal_Cond': '热导率 (W/m·K)',
                        'Cp': '比热容 (J/kg·K)'
                    })

fig.update_traces(textposition='top center')
fig.update_layout(height=600)
fig.show()

## 7. 机器学习预测 🤖

使用机器学习模型预测聚合物性质：

In [ ]:
# 准备机器学习数据集
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 生成更多的训练数据（使用分子描述符）
def generate_ml_dataset(n_samples=100):
    """生成用于ML的数据集"""
    data = []
    
    for i in range(n_samples):
        # 随机选择聚合物类型
        polymer_type = np.random.choice(list(polymers_db.keys()))
        mol = utils.mol_from_smiles(polymers_db[polymer_type]['smiles'])
        
        # 计算分子描述符
        descriptors = {
            'MW': Descriptors.ExactMolWt(mol),
            'HeavyAtoms': mol.GetNumHeavyAtoms(),
            'Bonds': mol.GetNumBonds(),
            'RotBonds': Descriptors.NumRotatableBonds(mol),
            'HBA': Descriptors.NumHAcceptors(mol),
            'HBD': Descriptors.NumHDonors(mol),
            'TPSA': Descriptors.TPSA(mol),
            'LogP': Descriptors.MolLogP(mol),
            'MolVol': Descriptors.MolMR(mol),
        }
        
        # 添加目标值（基于聚合物类型）
        base_props = {
            'PE': 0.95, 'PP': 0.90, 'PS': 1.05, 
            'PVC': 1.40, 'PMMA': 1.18, 'PTFE': 2.20
        }
        
        descriptors['Density'] = base_props[polymer_type] + np.random.normal(0, 0.05)
        descriptors['Polymer'] = polymer_type
        
        data.append(descriptors)
    
    return pd.DataFrame(data)

# 生成数据集
ml_data = generate_ml_dataset(200)
print(f"📊 生成了 {len(ml_data)} 个样本的ML数据集")
print(f"\n特征: {list(ml_data.columns[:-2])}")
print(f"目标: Density")

In [ ]:
# 训练机器学习模型
# 准备特征和目标
feature_cols = ['MW', 'HeavyAtoms', 'Bonds', 'RotBonds', 'HBA', 'HBD', 'TPSA', 'LogP', 'MolVol']
X = ml_data[feature_cols]
y = ml_data['Density']

# 分割数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 训练随机森林模型
print("🤖 训练随机森林模型...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# 预测
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# 评估
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"\n📈 模型性能:")
print(f"   训练集 R²: {train_r2:.3f}, MAE: {train_mae:.4f}")
print(f"   测试集 R²: {test_r2:.3f}, MAE: {test_mae:.4f}")

In [ ]:
# 可视化预测结果
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. 预测vs实际
ax1 = axes[0]
ax1.scatter(y_test, y_pred_test, alpha=0.6, s=50)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax1.set_xlabel('实际密度 (g/cm³)', fontsize=12)
ax1.set_ylabel('预测密度 (g/cm³)', fontsize=12)
ax1.set_title(f'预测 vs 实际 (R² = {test_r2:.3f})', fontsize=14)
ax1.grid(True, alpha=0.3)

# 2. 残差分布
ax2 = axes[1]
residuals = y_test - y_pred_test
ax2.hist(residuals, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('残差', fontsize=12)
ax2.set_ylabel('频数', fontsize=12)
ax2.set_title('残差分布', fontsize=14)
ax2.grid(True, alpha=0.3)

# 3. 特征重要性
ax3 = axes[2]
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
ax3.bar(range(len(importances)), importances[indices], color='lightgreen', edgecolor='black')
ax3.set_xticks(range(len(importances)))
ax3.set_xticklabels([feature_cols[i] for i in indices], rotation=45, ha='right')
ax3.set_xlabel('特征', fontsize=12)
ax3.set_ylabel('重要性', fontsize=12)
ax3.set_title('特征重要性', fontsize=14)
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 输出最重要的特征
print("\n🔑 最重要的3个特征:")
for i in range(3):
    idx = indices[i]
    print(f"   {i+1}. {feature_cols[idx]}: {importances[idx]:.3f}")

## 8. 交互式聚合物设计工具 🛠️

创建一个交互式工具来设计和预测新聚合物的性质：

In [ ]:
from ipywidgets import interact, widgets, VBox, HBox, Output
import IPython.display as display

# 创建交互式聚合物设计器
def polymer_designer():
    """交互式聚合物设计工具"""
    
    # 创建控件
    polymer_type = widgets.Dropdown(
        options=list(polymers_db.keys()),
        value='PE',
        description='聚合物类型:'
    )
    
    dp_slider = widgets.IntSlider(
        value=10,
        min=5,
        max=50,
        step=5,
        description='聚合度:'
    )
    
    tacticity = widgets.RadioButtons(
        options=['atactic', 'isotactic', 'syndiotactic'],
        value='atactic',
        description='立构规整度:'
    )
    
    temp_slider = widgets.FloatSlider(
        value=300,
        min=200,
        max=500,
        step=10,
        description='温度 (K):'
    )
    
    output = Output()
    
    def update_polymer(polymer_type, dp, tacticity, temp):
        with output:
            output.clear_output()
            
            # 生成聚合物
            print(f"🔬 生成 {polymers_db[polymer_type]['name']}...")
            print(f"   参数: DP={dp}, 立构={tacticity}, T={temp}K")
            
            # 创建分子
            mol = utils.mol_from_smiles(polymers_db[polymer_type]['smiles'])
            
            # 计算预测性质
            descriptors = [
                Descriptors.ExactMolWt(mol) * dp,
                mol.GetNumHeavyAtoms() * dp,
                mol.GetNumBonds() * dp,
                Descriptors.NumRotatableBonds(mol) * dp,
                Descriptors.NumHAcceptors(mol) * dp,
                Descriptors.NumHDonors(mol) * dp,
                Descriptors.TPSA(mol),
                Descriptors.MolLogP(mol),
                Descriptors.MolMR(mol) * dp
            ]
            
            # 使用训练好的模型预测
            X_new = np.array(descriptors).reshape(1, -1)
            predicted_density = rf_model.predict(X_new)[0]
            
            # 温度修正（简化模型）
            density_corrected = predicted_density * (1 - 0.0007 * (temp - 300))
            
            # 显示结果
            print(f"\n📊 预测性质:")
            print(f"   分子量: {descriptors[0]:.0f} g/mol")
            print(f"   原子数: {int(descriptors[1])}")
            print(f"   预测密度: {density_corrected:.3f} g/cm³")
            
            # 估算其他性质（基于经验关系）
            if tacticity == 'isotactic':
                crystallinity = 0.7
            elif tacticity == 'syndiotactic':
                crystallinity = 0.5
            else:
                crystallinity = 0.1
            
            print(f"   结晶度估计: {crystallinity*100:.0f}%")
            
            # 可视化分子
            polymer = poly.polymerize_rw(mol, min(dp, 10), tacticity=tacticity)
            polymer = poly.terminate_rw(polymer, utils.mol_from_smiles('*C'))
            
            # 显示2D结构
            img = Draw.MolToImage(polymer, size=(600, 200))
            display.display(img)
    
    # 创建交互界面
    interact(update_polymer, 
             polymer_type=polymer_type,
             dp=dp_slider,
             tacticity=tacticity,
             temp=temp_slider)
    
    display.display(output)

# 运行设计器
print("🛠️ 聚合物设计工具")
print("调整参数来设计你的聚合物！\n")
polymer_designer()

## 9. 总结与下一步 📝

### 本教程涵盖的内容：
- ✅ RadonPy 基础设置和导入
- ✅ 分子结构可视化
- ✅ 聚合物链生成
- ✅ 力场参数分析
- ✅ MD模拟监控
- ✅ 物性结果分析
- ✅ 机器学习预测
- ✅ 交互式设计工具

### 实际应用建议：

1. **开始简单**：先从小分子和短链开始
2. **验证结果**：与实验数据对比
3. **优化参数**：根据体系调整模拟参数
4. **批量计算**：使用并行计算提高效率
5. **数据管理**：建立自己的聚合物数据库

In [ ]:
# 保存工作成果
def save_results():
    """保存本次会话的结果"""
    import datetime
    import pickle
    
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 保存数据
    results = {
        'polymers_db': polymers_db,
        'polymer_chains': polymer_chains,
        'md_data': md_data,
        'polymer_props': polymer_props_df,
        'ml_model': rf_model,
        'timestamp': timestamp
    }
    
    # 保存为pickle
    with open(f'radonpy_session_{timestamp}.pkl', 'wb') as f:
        pickle.dump(results, f)
    
    # 保存数据表为CSV
    polymer_props_df.to_csv(f'polymer_properties_{timestamp}.csv', index=False)
    md_data.to_csv(f'md_trajectory_{timestamp}.csv', index=False)
    
    print(f"✅ 结果已保存:")
    print(f"   - radonpy_session_{timestamp}.pkl")
    print(f"   - polymer_properties_{timestamp}.csv")
    print(f"   - md_trajectory_{timestamp}.csv")
    
    return timestamp

# 保存结果
session_id = save_results()
print(f"\n🎉 教程完成！会话ID: {session_id}")

## 附录：有用的代码片段 💡

### 快速参考

In [ ]:
# 常用代码片段集合
quick_reference = """
# 1. 快速生成聚合物
mol = utils.mol_from_smiles('*CC(*)')
polymer = poly.polymerize_rw(mol, 20)

# 2. 力场分配
ff = GAFF2_mod()
ff.ff_assign(polymer)

# 3. 创建模拟单元
cell = poly.amorphous_cell(polymer, 10, density=0.05)

# 4. 运行平衡
eqmd = eq.EQ21step(cell, work_dir='./work')
cell = eqmd.exec(temp=300, press=1.0)

# 5. 分析结果
analy = eqmd.analyze()
props = analy.get_all_prop(temp=300, press=1.0)

# 6. 计算热导率
nemd = tc.NEMD_MP(cell, work_dir='./tc')
cell = nemd.exec(temp=300)
tc_analy = nemd.analyze()
tc = tc_analy.calc_tc()
"""

print("📚 快速参考代码:")
print(quick_reference)

---

### 🎓 恭喜完成 RadonPy 交互式教程！

如有问题，请访问：
- GitHub: https://github.com/RadonPy/RadonPy
- Issues: https://github.com/RadonPy/RadonPy/issues

Happy polymer modeling! 🧪🔬